In [ ]:
from langgraph.graph import StateGraph, START, MessagesState
from langgraph.checkpoint.memory import InMemorySaver
from langchain_ollama import ChatOllama
from langchain.messages import RemoveMessage

In [ ]:
model = ChatOllama(model="phi3")


In [ ]:
def chat(state: MessagesState):
    response = model.invoke(state["messages"])
    return {"messages": [response]}

def delete_old_messages(state: MessagesState):
    msgs = state["messages"]

    # if more than 10 messages, delete the earliest 6
    if len(msgs) > 10:
        to_remove = msgs[:6]
        return {"messages": [RemoveMessage(id=m.id) for m in to_remove]}

    return {}

In [ ]:
builder = StateGraph(MessagesState)
builder.add_node("chat", chat)
builder.add_node("cleanup", delete_old_messages)

In [ ]:
builder.add_edge(START, "chat")
builder.add_edge("chat", "cleanup")   # run deletion after each response
builder.add_edge("cleanup", "__end__")

In [ ]:
graph = builder.compile(checkpointer=InMemorySaver())


In [ ]:
graph


In [ ]:
config = {"configurable": {"thread_id": "t1"}}


In [ ]:
# Run multiple turns
graph.invoke({"messages": [{"role": "user", "content": "Hi, I'm Nitish"}]}, config)
graph.invoke({"messages": [{"role": "user", "content": "Tell me about LangGraph"}]}, config)
graph.invoke({"messages": [{"role": "user", "content": "Now explain checkpointers"}]}, config)
graph.invoke({"messages": [{"role": "user", "content": "What is Langchain"}]}, config)
graph.invoke({"messages": [{"role": "user", "content": "What is Quantum Mechanics"}]}, config)
graph.invoke({"messages": [{"role": "user", "content": "What is Gen AI"}]}, config)
graph.invoke({"messages": [{"role": "user", "content": "What is my name"}]}, config)

In [ ]:
snap = graph.get_state(config)
print("Stored messages after cleanup:", len(snap.values["messages"]))

In [ ]:
# Streaming version
def chat_stream(state: MessagesState):
    """Streaming chat node - yields chunks as they arrive"""
    for chunk in model.stream(state["messages"]):
        # Yield each chunk to the channel
        yield chunk

# Create a new graph with streaming
builder2 = StateGraph(MessagesState)
builder2.add_node("chat", chat_stream)
builder2.add_node("cleanup", delete_old_messages)

builder2.add_edge(START, "chat")
builder2.add_edge("chat", "cleanup")
builder2.add_edge("cleanup", "__end__")

graph_stream = builder2.compile(checkpointer=InMemorySaver())

In [ ]:
# Test streaming with a single query
print("Testing streaming...")
for chunk in graph_stream.stream(
    {"messages": [{"role": "user", "content": "What is AI?"}]},
    config
):
    print(chunk, end="", flush=True)

In [ ]:
# Run multiple turns with streaming
print("\n--- Running multiple turns with streaming ---")
graph_stream.invoke({"messages": [{"role": "user", "content": "Hi, I'm Nitish"}]}, config)
graph_stream.invoke({"messages": [{"role": "user", "content": "Tell me about LangGraph"}]}, config)
graph_stream.invoke({"messages": [{"role": "user", "content": "Now explain checkpointers"}]}, config)
graph_stream.invoke({"messages": [{"role": "user", "content": "What is Langchain"}]}, config)
graph_stream.invoke({"messages": [{"role": "user", "content": "What is Quantum Mechanics"}]}, config)
graph_stream.invoke({"messages": [{"role": "user", "content": "What is Gen AI"}]}, config)
graph_stream.invoke({"messages": [{"role": "user", "content": "What is my name"}]}, config)

# Check stored messages after cleanup
snap = graph_stream.get_state(config)
print("\nStored messages after cleanup:", len(snap.values["messages"]))